# Develop acceptance loop

In [2]:
from sklearn.ensemble import IsolationForest
import torch

import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataGenerator, CreditData
from berebasl.estimation.basl import BASLPartialUnbiaser, accept_based_on_top_percentent_of_arbitrary_var
from berebasl.estimation.bayesian_evaluation import BayesianMetric, batched_auroc
from berebasl.estimation.classifiers import TorchLogistic

In [ ]:
dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device(
    "cuda" if torch.cuda.is_available() else 
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else 
    "cpu"
)

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 200
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0], dtype=dtype, device=device),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]], dtype=dtype, device=device),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]], dtype=dtype, device=device)
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    dtype=dtype
)

# Initial population
data_gen.manual_seed(initial_seed)
features, default_flag = data_gen.sample(init_sample)

accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features, 
    default_flag,
    var_for_rule=0,
    top_percent=top_percent,
    default_value = CreditDataGenerator.bad_good_encoding["bad"]
)

credit_data = CreditData(features, default_flag, accepts)

# Holdout Population
data_gen.manual_seed(-initial_seed)
holdout_features, holdout_flag = data_gen.sample(n=holdout_sample)
holdout_data = CreditData(
    holdout_features, holdout_flag, 
    accepted_initial=torch.ones(holdout_flag.shape, dtype=torch.bool) # All are "accepted"
)

strong_learner = TorchLogistic(n_features=credit_data.features_count)
# BASL related classes
basl_unbiaser = BASLPartialUnbiaser(
    filtering_quantiles={"lower" : 0.01, "upper" : 0.99},
    weak_learner=TorchLogistic(n_features=credit_data.features_count),
    strong_learner=strong_learner,
    holdout_percent=0.1,
    sampling_percent=0.8,
    label_bads_percent=0.1,
    label_goods_percent=0.1/2,
    max_iterations=5,
    early_stop=True,
    isolation_forest=IsolationForest(n_estimators=100, max_samples="auto", random_state=1807),
    bayesian_metric = BayesianMetric(
        model=strong_learner,
        min_iterations=1e2,
        max_iterations=1e5,
        epsilon=1e-5,
        metric = batched_auroc,
        device=credit_data.device
    )
)

# Acceptance Loop

stats = []

for gen_nr in range(1, num_gens + 1):
    if gen_nr % 10 == 0:
        print("-- Iteration", f"{gen_nr}/{num_gens}:", credit_data.accepted_count, 
              "accepts and", credit_data.rejected_count, " rejects")
        
    ## Gather current statistics
    current_stats : dict = credit_data.data_stats()

    ## Generate new data
    data_gen.manual_seed(initial_seed + gen_nr)
    features, default_flag = data_gen.sample(sample_size)

    current_sample = credit_data.to_sample_dataset()

    ## Accepts based scorecard




sel